# Impact Fund Name Screener

Identifies funds whose name suggests an impact mandate via regex matching against Dirk's keyword list (English + multilingual equivalents).

**Run:** Kernel → Restart & Run All  
**Edit:** Cell 1 (paths/columns) and Cell 2 (patterns) only.

## CELL 1 — Configuration

Edit paths and column list here. All other cells run without changes.

In [1]:
import os, re, json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

config = {}
with open("File_Directory.txt") as f:
    for line in f:
        if ":" in line:
            key, val = line.split(":", 1)
            config[key.strip()] = val.strip()

INPUT_FILE  = Path(config["Input"])
OUTPUT_DIR  = Path(config["Output"])

NAME_COL    = "Name"
ID_COL      = "FundId"

OBJECTIVE_COLUMNS = [
    "Prospectus Objective",
    "KIID Objective/Investment Policy",
    "PRIIPS KID Objective",
    "Strategy Description",
    "Investment Strategy - English",
    "PRIIPS KID Objective - Danish",
    "PRIIPS KID Objective - Dutch",
    "PRIIPS KID Objective - Finnish",
    "PRIIPS KID Objective - French",
    "PRIIPS KID Objective - German",
    "PRIIPS KID Objective - Italian",
    "PRIIPS KID Objective - Norwegian",
    "PRIIPS KID Objective - Portuguese",
    "PRIIPS KID Objective - Spanish",
    "PRIIPS KID Objective - Swedish",
    "KIID Objective/Investment Policy - German",
    "KIID Objective/Investment Policy - French",
    "KIID Objective/Investment Policy - Italian",
    "KIID Objective/Investment Policy - Spanish",
    "KIID Objective/Investment Policy - Norwegian",
    "KIID Objective/Investment Policy - Swedish",
    "KIID Objective/Investment Policy - Finnish",
    "KIID Objective/Investment Policy - Portuguese",
    "KIID Objective/Investment Policy - Danish",
    "Investment Strategy - Danish",
    "Investment Strategy - Finnish",
    "Investment Strategy - French",
    "Investment Strategy - German",
    "Investment Strategy - Italian",
    "Investment Strategy - Norwegian",
    "Investment Strategy - Portuguese",
    "Investment Strategy - Spanish",
    "Investment Strategy - Swedish",
]

print("Configuration loaded.")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_DIR}")


Configuration loaded.
  Input:  /Users/dannyhogan/Desktop/Hogan_RA_Work/Download Sustainable Funds 2026-04-15.xlsx
  Output: /Users/dannyhogan/Desktop/Hogan_RA_Work


## CELL 2 — Keyword Patterns

One entry per `(label, language, regex, needs_review)`. `needs_review=True` flags the match for human inspection — use for ambiguous abbreviations or high false-positive-risk terms.

In [2]:
# Each entry: (label, language, regex_pattern, needs_review)
#
# needs_review=True  → match is flagged for human inspection
#                       because the abbreviation is ambiguous
#                       or the term has false-positive risk.
# ============================================================

# re.IGNORECASE applied at compile time for all patterns
FLAGS = re.IGNORECASE

PATTERNS = [

    # ── English ────────────────────────────────────────────
    ("impact",                  "EN", r"\bimpact\b",                                         False),
    ("positive_change",         "EN", r"\bpositive[\s\-]change\b",                           False),
    ("better_world",            "EN", r"\bbetter[\s\-]world\b",                              False),
    ("future_generations",      "EN", r"\bfuture(?:[\s\-]for)?[\s\-]generations?\b",         False),
    ("generations_abbrev",      "EN", r"\bGens\b",                                           True),   # e.g. "Food For Gens"
    ("sdg",                     "EN", r"\bSDGs?\b",                                          False),
    ("sustainable_development", "EN", r"\bsustainable[\s\-]dev(?:elopment)?\b",              False),
    ("sust_dev_abbrev",         "EN", r"\bSust(?:[\s\-]Dev\b|\b)",                           True),   # Sust alone is ambiguous
    ("transformation",          "EN", r"\btransform(?:ation|ative|ations)?\b",               False),
    ("transition",              "EN", r"\btransition\b",                                     False),
    ("trans_abbrev",            "EN", r"\bTrans\b",                                          True),   # Trans alone is ambiguous
    ("climate_action",          "EN", r"\bclimate[\s\-]action\b",                            False),  # full phrase only
    ("engagement",              "EN", r"\bengagement\b",                                     False),
    ("stewardship",             "EN", r"\bstewards?(?:hip)?\b",                              False),

    # ── French ─────────────────────────────────────────────
    ("impact_fr",               "FR", r"\bimpact\b",                                         False),
    ("transition_fr",           "FR", r"\btransition\b",                                     False),
    ("transformation_fr",       "FR", r"\btransformation\b",                                 False),
    ("dev_durable",             "FR", r"\bd[ée]veloppement[\s\-]durable\b",                  False),
    ("action_climatique",       "FR", r"\baction[\s\-]climatique\b",                         False),  # full phrase — not bare "action"
    ("engagement_fr",           "FR", r"\bengagement\b",                                     False),
    ("isr",                     "FR", r"\bISR\b",                                            False),  # Investissement Socialement Responsable
    ("generations_futures",     "FR", r"\bg[ée]n[ée]rations?[\s\-]futures?\b",               False),
    ("meilleur_monde",          "FR", r"\bmeilleur[\s\-]monde\b",                            False),
    ("changement_positif",      "FR", r"\bchangement[\s\-]positif\b",                        False),

    # ── German ─────────────────────────────────────────────
    ("impact_de",               "DE", r"\bimpact\b",                                         False),
    ("transformation_de",       "DE", r"\btransformation\b",                                 False),
    ("transition_de",           "DE", r"\btransition\b",                                     False),
    ("nachhaltige_entwicklung", "DE", r"\bnachhaltige[\s\-]entwicklung\b",                   False),
    ("klimaschutz",             "DE", r"\bklimaschutz\b",                                    False),
    ("klimaaktion",             "DE", r"\bklima(?:aktion|schutz)\b",                         False),
    ("engagement_de",           "DE", r"\bengagement\b",                                     False),
    ("generationen",            "DE", r"\bgenerationen\b",                                   False),
    ("positiver_wandel",        "DE", r"\bpositiver?[\s\-]wandel\b",                         False),
    ("bessere_welt",            "DE", r"\bbessere[\s\-]welt\b",                              False),

    # ── Spanish ────────────────────────────────────────────
    ("impacto",                 "ES", r"\bimpacto\b",                                        False),
    ("transicion",              "ES", r"\btransici[oó]n\b",                                  False),
    ("transformacion",          "ES", r"\btransformaci[oó]n\b",                              False),
    ("desarrollo_sostenible",   "ES", r"\bdesarrollo[\s\-]sostenible\b",                     False),
    ("accion_climatica",        "ES", r"\bacci[oó]n[\s\-]clim[aá]tica\b",                   False),  # full phrase — not bare "acción"
    ("compromiso",              "ES", r"\bcompromiso\b",                                     True),   # "engagement" but also general; review
    ("generaciones_futuras",    "ES", r"\bgeneraciones?[\s\-]futuras?\b",                    False),
    ("ods",                     "ES", r"\bODS\b",                                            False),  # Objetivos de Desarrollo Sostenible
    ("mejor_mundo",             "ES", r"\bmejor[\s\-]mundo\b",                               False),

    # ── Italian ────────────────────────────────────────────
    ("impatto",                 "IT", r"\bimpatto\b",                                        False),
    ("transizione",             "IT", r"\btransizione\b",                                    False),
    ("trasformazione",          "IT", r"\btrasformazione\b",                                 False),
    ("sviluppo_sostenibile",    "IT", r"\bsviluppo[\s\-]sostenibile\b",                      False),
    ("azione_climatica",        "IT", r"\bazione[\s\-]climatica\b",                          False),
    ("generazioni_future",      "IT", r"\bgenerazioni[\s\-]future\b",                        False),

    # ── Swedish ────────────────────────────────────────────
    ("omstallning",             "SV", r"\bomst[äa]llning\b",                                 False),  # transition
    ("hallbar_utveckling",      "SV", r"\bh[äa]llbar[\s\-]utveckling\b",                     False),  # sustainable development
    ("klimataktion",            "SV", r"\bklimat(?:aktion|akt)\b",                           False),
    ("engagement_sv",           "SV", r"\bengagemang\b",                                     False),
    ("framtida_generationer",   "SV", r"\bframtida[\s\-]generationer\b",                     False),

    # ── Danish / Norwegian ─────────────────────────────────
    ("baeredygtighed",          "DA/NO", r"\bb[æa]redygtighed\b",                            False),  # sustainability (DA)
    ("overgang",                "DA/NO", r"\bovergang\b",                                    True),   # transition (also means "crossing"); review
    ("ans_tran_abbrev",         "DA/NO", r"\bAnsTran\b",                                     True),   # likely "Ansvarlig Transition"; review

    # ── Dutch ──────────────────────────────────────────────
    ("impact_nl",               "NL", r"\bimpact\b",                                         False),
    ("overgang_nl",             "NL", r"\bovergang\b",                                       True),   # transition; review
    ("duurzame_ontwikkeling",   "NL", r"\bduurzame[\s\-]ontwikkeling\b",                     False),
    ("klimaatactie",            "NL", r"\bklimaatactie\b",                                   False),
]

# Compile patterns
COMPILED_PATTERNS = [
    (label, lang, re.compile(pat, FLAGS), review)
    for label, lang, pat, review in PATTERNS
]

print(f"Loaded {len(COMPILED_PATTERNS)} patterns across "
      f"{len(set(lang for _, lang, _, _ in PATTERNS))} language groups.")


Loaded 61 patterns across 8 language groups.


## CELL 3 — Abbreviation Expansion Map

Token-level expansion applied to matched fund names only. Best-effort: expands known tokens, flags residual unknowns in `Expansion_Complete` column.

In [3]:
# Applied to matched tokens only — not full unabbreviation
# of the entire fund name (too error-prone at scale).
# ============================================================

# Token-level expansion: token → expanded form
# Tokens must match exactly (case-sensitive after stripping)
ABBREV_EXPANSION = {
    "Sust":    "Sustainable",
    "Sus":     "Sustainable",
    "Dev":     "Development",
    "Gens":    "Generations",
    "Clmt":    "Climate",
    "Clim":    "Climate",
    "Env":     "Environmental",
    "Soc":     "Social",
    "Gov":     "Governance",
    "Eq":      "Equity",          # most common meaning
    "Fds":     "Funds",
    "Fd":      "Fund",
    "Mkt":     "Market",
    "Mkts":    "Markets",
    "Intl":    "International",
    "Intern":  "International",
    "Glb":     "Global",
    "Glbl":    "Global",
    "Emg":     "Emerging",
    "Em":      "Emerging",
    "Cnsrv":   "Conservative",
    "Eqs":     "Equities",
}

# Tokens that are KNOWN negatives (Trans → Transportation)
# If the matched abbreviation appears in this context, downgrade to review
KNOWN_NEGATIVE_ABBREVS = {
    "Transp": "Transportation",
    "TransP": "Transportation",
}

def expand_name(fund_name: str) -> tuple[str, bool]:
    """
    Best-effort token expansion. Returns (expanded_name, fully_expanded).
    fully_expanded=False if any token could not be confidently expanded.
    """
    tokens = re.split(r'(\s+)', fund_name)  # preserve whitespace
    expanded_tokens = []
    fully_expanded = True
    for token in tokens:
        stripped = token.strip()
        if stripped in ABBREV_EXPANSION:
            expanded_tokens.append(ABBREV_EXPANSION[stripped])
        elif stripped in KNOWN_NEGATIVE_ABBREVS:
            expanded_tokens.append(KNOWN_NEGATIVE_ABBREVS[stripped])
        elif re.match(r'^[A-Z]{2,5}$', stripped) and stripped not in {
            "EUR", "USD", "GBP", "CHF", "JPY", "SEK", "NOK", "DKK",  # currencies
            "ETF", "UCITS", "ELTIF",                                    # fund types
            "ESG", "ISR", "SDG", "ODS", "ODD",                         # already-known acronyms
            "ACC", "INC", "CAP", "DIS",                                 # share class
            "UK", "US", "EU", "EM", "EMU",                              # geography
            "AI", "IT", "IP",                                           # tech / share class
        }:
            # Uppercase token we don't know → mark as incomplete
            expanded_tokens.append(token)
            fully_expanded = False
        else:
            expanded_tokens.append(token)
    return "".join(expanded_tokens), fully_expanded


## CELL 4 — Token Scan (exploratory)

Exploratory. Prints uppercase abbreviations in the full dataset not in the known-safe set. Run once to check for gaps in `ABBREV_EXPANSION` before committing to the full matching pass.

In [4]:
# Run once on the full dataset to surface unknown abbreviations
# before finalising the ABBREV_EXPANSION map.
# ============================================================

print("Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"  {len(df)} funds, {len(df.columns)} columns")

all_tokens = []
for name in df[NAME_COL].dropna():
    tokens = re.split(r'[\s\-–&/()+]+', str(name))
    all_tokens.extend(t for t in tokens if t)

token_counts = Counter(all_tokens)

# Candidates: 2–5 char uppercase, not in known-safe set, appearing ≥ 2 times
SAFE_TOKENS = {
    "EUR", "USD", "GBP", "CHF", "JPY", "SEK", "NOK", "DKK", "HKD",
    "ETF", "UCITS", "ELTIF", "AIF",
    "ESG", "ISR", "SDG", "ODS", "ODD",
    "ACC", "INC", "CAP", "DIS",
    "UK", "US", "EU", "EM", "EMU", "USA",
    "AI", "IT", "IP", "KID", "KIID",
    "FI",   # Spanish: Fondo de Inversión
    "SI",   # share class identifier
    "A", "B", "C", "D", "I", "R", "Z",   # single-letter share classes
}

unknown_abbrevs = [
    (tok, count)
    for tok, count in token_counts.most_common(500)
    if re.match(r'^[A-Z]{2,5}$', tok)
    and tok not in SAFE_TOKENS
    and count >= 2
]

print(f"\n{'Token':<12} {'Count':>6}   (possible meaning)")
print("─" * 45)
for tok, count in unknown_abbrevs[:40]:
    known = ABBREV_EXPANSION.get(tok, "?")
    print(f"{tok:<12} {count:>6}   {known}")


Loading data...
  5680 funds, 133 columns

Token         Count   (possible meaning)
─────────────────────────────────────────────
AM               77   ?
JPM              75   ?
KL               72   ?
GS               70   ?
DWS              67   ?
BNP              66   ?
SICAV            65   ?
IC               61   ?
ISF              56   ?
AXA              51   ?
UBS              51   ?
QI               50   ?
CM               47   ?
SRI              45   ?
PME              45   ?
SEB              45   ?
HSBC             41   ?
LUX              41   ?
IM               40   ?
CPR              40   ?
II               38   ?
BGF              38   ?
KBC              37   ?
CT               36   ?
MS               35   ?
INVF             35   ?
ES               34   ?
SG               34   ?
DNB              34   ?
AB               32   ?
AZ               31   ?
BI               30   ?
RC               29   ?
MFS              29   ?
EF               28   ?
AC               26   ?
DPAM  

## CELL 5 — Matching Function

Core matching logic. Returns all pattern hits for a single fund name string.

In [5]:
def match_fund(fund_name: str) -> list[dict]:
    """
    Returns a list of match records for a single fund name.
    One record per pattern matched (a name may match multiple patterns).
    """
    matches = []
    for label, lang, compiled_re, needs_review in COMPILED_PATTERNS:
        m = compiled_re.search(fund_name)
        if m:
            matched_text = m.group(0)
            matches.append({
                "matched_pattern_label": label,
                "matched_language":      lang,
                "matched_text":          matched_text,
                "needs_review":          needs_review,
            })
    return matches


## CELL 6 — Run Matching

Applies `match_fund()` across all funds. Attaches all objective and strategy text columns to each match row.

In [6]:
print("Running pattern matching...")

# Only keep objective columns that exist in this dataset
available_obj_cols = [c for c in OBJECTIVE_COLUMNS if c in df.columns]

results = []

for _, row in df.iterrows():
    fund_name = str(row.get(NAME_COL, ""))
    fund_id   = row.get(ID_COL, "")

    matches = match_fund(fund_name)
    if not matches:
        continue

    expanded_name, fully_expanded = expand_name(fund_name)

    # Collect all objective/strategy text for output columns
    obj_texts = {col: row.get(col, "") for col in available_obj_cols}

    # One row per matched pattern
    for match in matches:
        results.append({
            ID_COL:              fund_id,
            NAME_COL:            fund_name,
            "Name_Expanded":     expanded_name,
            "Expansion_Complete": fully_expanded,
            **match,
            **obj_texts,
        })

results_df = pd.DataFrame(results)
print(f"  {results_df[ID_COL].nunique()} funds matched across {len(results_df)} pattern hits")


Running pattern matching...
  337 funds matched across 596 pattern hits


## CELL 7 — Summary Statistics

Printed console summary: total candidates, hit counts by pattern label and language group, review flag count.

In [7]:
if len(results_df) == 0:
    print("No matches found.")
else:
    # Deduplicate to one row per fund for counting
    funds_df = results_df.drop_duplicates(subset=ID_COL)

    print(f"\n{'═'*55}")
    print(f"  IMPACT FUND CANDIDATES: {len(funds_df)} of {len(df)} funds")
    print(f"  ({len(funds_df)/len(df)*100:.1f}% of full dataset)")
    print(f"{'═'*55}")

    print(f"\nHits by pattern label:")
    for label, count in results_df["matched_pattern_label"].value_counts().items():
        n_review = results_df[results_df["matched_pattern_label"] == label]["needs_review"].sum()
        flag = "  ⚠ needs review" if n_review > 0 else ""
        print(f"  {label:<30} {count:>4}{flag}")

    print(f"\nHits by language group:")
    for lang, count in results_df["matched_language"].value_counts().items():
        print(f"  {lang:<10} {count:>4}")

    review_count = results_df["needs_review"].sum()
    print(f"\nRows flagged for human review: {review_count} "
          f"({review_count/len(results_df)*100:.1f}% of all hits)")



═══════════════════════════════════════════════════════
  IMPACT FUND CANDIDATES: 337 of 5680 funds
  (5.9% of full dataset)
═══════════════════════════════════════════════════════

Hits by pattern label:
  sust_dev_abbrev                 107  ⚠ needs review
  isr                              89
  impact                           57
  impact_fr                        57
  impact_de                        57
  impact_nl                        57
  transition                       35
  transition_fr                    35
  transition_de                    35
  sdg                              17
  engagement                        5
  engagement_fr                     5
  engagement_de                     5
  trans_abbrev                      5  ⚠ needs review
  climate_action                    4
  better_world                      3
  dev_durable                       2
  future_generations                2
  impacto                           2
  transformation                    2
  

## CELL 8 — Output to Excel

Writes four sheets: **Matches** (one row per hit), **Funds_Deduped** (one row per fund), **Summary** (pattern-level stats), **Token_Scan** (unknown abbreviation candidates).

In [8]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
outfile = OUTPUT_DIR / f"Impact_Fund_Candidates_{timestamp}.xlsx"

with pd.ExcelWriter(outfile, engine="openpyxl") as writer:

    # Sheet 1 — Full results (one row per pattern hit)
    results_df.to_excel(writer, sheet_name="Matches", index=False)

    # Sheet 2 — One row per fund (first/most-significant match)
    if len(results_df) > 0:
        deduped = (
            results_df
            .sort_values("needs_review")          # confirmed hits first
            .drop_duplicates(subset=ID_COL, keep="first")
        )
        deduped.to_excel(writer, sheet_name="Funds_Deduped", index=False)

    # Sheet 3 — Summary statistics
    if len(results_df) > 0:
        summary_rows = []
        for label, grp in results_df.groupby("matched_pattern_label"):
            lang = grp["matched_language"].iloc[0]
            n_funds = grp[ID_COL].nunique()
            n_review = int(grp["needs_review"].sum())
            example = grp[NAME_COL].iloc[0]
            summary_rows.append({
                "Pattern Label":    label,
                "Language":         lang,
                "Funds Matched":    n_funds,
                "Needs Review (n)": n_review,
                "Example Name":     example,
            })
        summary_df = pd.DataFrame(summary_rows).sort_values("Funds Matched", ascending=False)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)

    # Sheet 4 — Token scan: unknown abbreviations from Cell 4
    abbrev_rows = [
        {"Token": tok, "Count": count, "Known Expansion": ABBREV_EXPANSION.get(tok, "UNKNOWN")}
        for tok, count in unknown_abbrevs[:100]
    ]
    pd.DataFrame(abbrev_rows).to_excel(writer, sheet_name="Token_Scan", index=False)

print(f"\nOutput written to:\n  {outfile}")



Output written to:
  /Users/dannyhogan/Desktop/Hogan_RA_Work/Impact_Fund_Candidates_20260619_1400.xlsx
